In [ ]:
#instalando a biblioteca
!pip install alpha_vantage

In [ ]:
# chamando as biblotecas
from alpha_vantage.timeseries import TimeSeries
import requests
import pandas as pd
import json
import numpy as np

#Declando variaveis
api_key = 'DGLLCB818CWYTWLU'
symbol = 'AAPL'
url = 'https://www.alphavantage.co/query?function=TIME_SERIES_MONTHLY&symbol={symbol}&apikey={api_key}'
r = requests.get(url)

In [ ]:
data = r.json()['Monthly Time Series']

In [ ]:
#Importando os dados temporais mensais para um Data Frame
df = data

#Renomiando as colunas e organizando os tipos de dados das colunas
df = pd.DataFrame(data).T.reset_index()
df.columns = ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']
df = df.astype({'Date': 'datetime64[ns]', 'Open': 'float', 'High': 'float', 'Low': 'float', 'Close': 'float', 'Volume': 'float'})


In [ ]:
# Lista de colunas numéricas
numeric_cols = ['Open', 'High', 'Low', 'Close', 'Volume']

# Aplicando o método IQR em todas as colunas
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    # Mantendo apenas os valores dentro dos limites
    df = df[(df[col] >= lower_bound) & (df[col] <= upper_bound)]

In [ ]:
#Exportar em .parquet
df.to_parquet('timeseries_monthly_APPLE.parquet', engine='pyarrow')